# ERC-20: collinearity-aware feature selection

Section 4.1. Many descriptors are near-duplicates (for instance `total_transactions` and `num_transactions`). Correlated inputs blur SHAP attributions, so before training the explained model we reduce the feature set on the CA + CMC scenario:

1. keep the mean of every node- and edge-level descriptor and all scalar descriptors, and drop constant columns;
2. drop one member of every pair with $|\rho| \ge 0.999$ (perfect aliases);
3. repeatedly take the most correlated remaining pair and drop the member less correlated with the label, until no pair has $|\rho| \ge 0.7$.

Correlations are computed on the training partition only. At every step of 3 a Balanced Random Forest is refit to track how much accuracy the pruning costs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from bfx.modeling import SEED, harmonise

DATA = ROOT / "data" / "erc20"
RESULTS = ROOT / "results" / "erc20"
RESULTS.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "CA": ("chainabuse_scam_tokens", 1),
    "TO": ("tokenscout_scam_tokens", 1),
    "CMC": ("cmc_licit_tokens", 0),
    "TS": ("tokenscout_licit_tokens", 0),
}


def load(name):
    file, _ = SOURCES[name]
    df = pd.read_csv(DATA / "features" / f"{file}.csv")
    address = df["token_file"].str.removesuffix(".csv").str.rsplit("_", n=1).str[-1].str.lower()
    return pd.concat([df, address.rename("address")], axis=1)


def scenario(fraud, licit):
    data = harmonise({fraud: load(fraud), licit: load(licit)}, fraud_sources=[fraud], key="address")
    X = data.drop(columns=["token_file", "label", "source_directory", "address", "source", "y"], errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    limit = np.finfo(np.float32).max  # tree models work in float32
    return X.clip(-limit, limit), data["y"]

In [ ]:
X, y = scenario("CA", "CMC")
X = X.loc[:, X.nunique() > 1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f"{len(X):,} tokens ({y.sum():,} scam, {(y == 0).sum():,} legitimate), {X.shape[1]} non-constant columns")

In [ ]:
target = X_train.corrwith(y_train).abs().fillna(0.0)


def most_correlated(columns):
    corr = X_train[columns].corr().abs().fillna(0.0)
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
    (a, b), rho = upper.idxmax(), upper.max()
    weaker = a if target[a] < target[b] else b
    return weaker, (b if weaker == a else a), rho


stat = ("_std", "_min", "_max")
features = [c for c in X.columns if not c.endswith(stat)]
print(f"step 1: {len(features)} features")

aliases = []
while True:
    drop, keep, rho = most_correlated(features)
    if rho < 0.999:
        break
    aliases.append({"dropped": drop, "kept": keep, "rho": rho})
    features.remove(drop)
print(f"step 2: {len(features)} features after removing {len(aliases)} aliases")
pd.DataFrame(aliases)

In [ ]:
def fit_eval(columns):
    m = BalancedRandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1)
    pred = m.fit(X_train[columns], y_train).predict(X_test[columns])
    return balanced_accuracy_score(y_test, pred), f1_score(y_test, pred)


log = [{"n_features": len(features), "dropped": None, "kept": None, "rho": None,
        **dict(zip(("balanced_accuracy", "f1"), fit_eval(features)))}]
while len(features) > 1:
    drop, keep, rho = most_correlated(features)
    if rho < 0.7:
        break
    features.remove(drop)
    log.append({"n_features": len(features), "dropped": drop, "kept": keep, "rho": rho,
                **dict(zip(("balanced_accuracy", "f1"), fit_eval(features)))})

log = pd.DataFrame(log)
log.to_csv(RESULTS / "pruning_log.csv", index=False)
pd.Series(features, name="feature").to_csv(RESULTS / "selected_features.csv", index=False)
print(f"step 3: {len(features)} features")
log.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(log["n_features"], log["balanced_accuracy"], "o-", label="balanced accuracy")
ax.plot(log["n_features"], log["f1"], "s-", label="F1 (scam)")
ax.invert_xaxis()
ax.set_xlabel("features kept")
ax.legend(frameon=False)
fig.savefig(RESULTS / "pruning_curve.png", dpi=300, bbox_inches="tight")

Selected features:

In [ ]:
pd.DataFrame({"feature": features, "|corr with label|": target[features].round(3).to_numpy()})